[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github.com/MLinApp-polito/mla-prj-23-project-am04_group-am01/blob/main/defect_detection.ipynb)

TODO: fix the button

# PBF Defect Detection

This notebook covers data loading, training, validation, and inference for detecting defects in Powder Bed Fusion images using a fine-tuned CNN.

## Clone GithHub repo

In [ ]:
!rm -rf mla_project/

In [6]:
import os

if not os.path.exists("/content/mla-prj-23-project-am04_group-am01") and not os.path.exists("/content/mla_project"):
  # DON'T SHARE THE PERSONAL ACCESS TOKEN

  # change the name of the branch here as needed
  !git clone -b gan https://***REMOVED-GITHUB-TOKEN***@github.com/MLinApp-polito/mla-prj-23-project-am04_group-am01.git

  # Rename folder for simplicity
  !mv /content/mla-prj-23-project-am04_group-am01 /content/mla_project

!cd /content/mla_project && git pull

remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 24 (delta 13), reused 21 (delta 10), pack-reused 0 (from 0)
Unpacking objects: 100% (24/24), 2.84 KiB | 485.00 KiB/s, done.
From https://github.com/MLinApp-polito/mla-prj-23-project-am04_group-am01
   4591c87..2c39edd  gan        -> origin/gan
   2b05faa..3865725  main       -> origin/main
Updating 4591c87..2c39edd
Fast-forward
 external/PyTorch-GAN/implementations/dcgan/dcgan.py | 7 ++++---
 1 file changed, 4 insertions(+), 3 deletions(-)


## Install Dependencies

In [4]:
!pip install torch torchvision matplotlib tqdm

## Imports

In [5]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Original dataset

## Dataset mean and std

In [ ]:
!python /content/mla_project/src/data_loader.py --data-dir /content/mla_project/images --compute-stats

Images shape:  torch.Size([16, 1, 1024, 1280])
Images shape:  torch.Size([16, 1, 1024, 1280])
Images shape:  torch.Size([16, 1, 1024, 1280])
Images shape:  torch.Size([16, 1, 1024, 1280])
Images shape:  torch.Size([10, 1, 1024, 1280])
Dataset mean (grayscale): 0.5830
Dataset std (grayscale): 0.2075


## Training - no augmentation

**IMPORTANT:**

- To perform K-Fold cross-validation, set --is_kfold to "True" and specify the number of folds with --k-folds. Example: --is_kfold "True", --k-folds 5

- To perform a single train/val split, set --is_kfold to "False" and specify the validation split ratio with --val-split. Example: --is_kfold "False", --val-split 0.2

- In both cases, to perform also testing, set --test to "True" and specify the test split ratio with --test-split. Example: --test "True", --test-split 0.2

### Define paths and parameters

In [ ]:
data_dir = '/content/mla_project/images'
train_dir = os.path.join(data_dir, 'train')
val_dir   = os.path.join(data_dir, 'val')

# Training params
batch_size = 1
epochs = 10
learning_rate = 1e-3
backbone = 'resnet50'
device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


### Launch training

In [ ]:
# k-fold cross validation (with test)

!python /content/mla_project/src/train.py \
    --data-dir "{data_dir}" \
    --batch-size {batch_size} \
    --epochs {10} \
    --lr {learning_rate} \
    --backbone {backbone} \
    --num-workers 2 \
    --is_kfold "True" \
    --k-folds 5 \
    --test "True" \
    --test-split 0.2

### Plot Training & Validation Curves

In [ ]:
# plot for cross-validation
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Read logs
logs = pd.read_csv('/content/kfold_logs.csv')

# Group for epoch and get mean and std
grouped = logs.groupby('epoch').agg({
    'train_loss': ['mean', 'std'],
    'val_loss': ['mean', 'std'],
    'train_acc': ['mean', 'std'],
    'val_acc': ['mean', 'std']
}).reset_index()

# Rename columns
grouped.columns = ['epoch',
                   'train_loss_mean', 'train_loss_std',
                   'val_loss_mean', 'val_loss_std',
                   'train_acc_mean', 'train_acc_std',
                   'val_acc_mean', 'val_acc_std']

# Set style
sns.set(style="white", context="notebook")

# LOSS
plt.figure(figsize=(8, 5))
plt.plot(grouped['epoch'], grouped['train_loss_mean'], label='Train Loss', color='blue')
plt.fill_between(grouped['epoch'],
                 grouped['train_loss_mean'] - grouped['train_loss_std'],
                 grouped['train_loss_mean'] + grouped['train_loss_std'],
                 color='blue', alpha=0.2)

plt.plot(grouped['epoch'], grouped['val_loss_mean'], label='Val Loss', color='orange')
plt.fill_between(grouped['epoch'],
                 grouped['val_loss_mean'] - grouped['val_loss_std'],
                 grouped['val_loss_mean'] + grouped['val_loss_std'],
                 color='orange', alpha=0.2)

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss (mean ± std)')
plt.legend()
sns.despine()
plt.tight_layout()
plt.show()

# ACCURACY
plt.figure(figsize=(8, 5))
plt.plot(grouped['epoch'], grouped['train_acc_mean'], label='Train Accuracy', color='green')
plt.fill_between(grouped['epoch'],
                 grouped['train_acc_mean'] - grouped['train_acc_std'],
                 grouped['train_acc_mean'] + grouped['train_acc_std'],
                 color='green', alpha=0.2)

plt.plot(grouped['epoch'], grouped['val_acc_mean'], label='Val Accuracy', color='red')
plt.fill_between(grouped['epoch'],
                 grouped['val_acc_mean'] - grouped['val_acc_std'],
                 grouped['val_acc_mean'] + grouped['val_acc_std'],
                 color='red', alpha=0.2)

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy (mean ± std)')
plt.legend()
sns.despine()
plt.tight_layout()
plt.show()


## Training - basic augmentations (simple transformations)

In [ ]:
data_dir = '/content/mla_project/images'
train_dir = os.path.join(data_dir, 'train')
val_dir   = os.path.join(data_dir, 'val')

# Training params
batch_size = 1
epochs = 10
learning_rate = 1e-3
backbone = 'resnet50'
device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# k-fold cross validation (with test) with augmented data

!python /content/mla_project/src/train.py \
    --data-dir "{data_dir}" \
    --batch-size {batch_size} \
    --epochs {10} \
    --lr {learning_rate} \
    --backbone {backbone} \
    --num-workers 2 \
    --is_kfold "True" \
    --k-folds 5 \
    --test "True" \
    --test-split 0.2 \
    --aug "True"

In [ ]:
# plot for cross-validation
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Read logs
logs = pd.read_csv('/content/kfold_logs.csv')

# Group for epoch and get mean and std
grouped = logs.groupby('epoch').agg({
    'train_loss': ['mean', 'std'],
    'val_loss': ['mean', 'std'],
    'train_acc': ['mean', 'std'],
    'val_acc': ['mean', 'std']
}).reset_index()

# Rename columns
grouped.columns = ['epoch',
                   'train_loss_mean', 'train_loss_std',
                   'val_loss_mean', 'val_loss_std',
                   'train_acc_mean', 'train_acc_std',
                   'val_acc_mean', 'val_acc_std']

# Set style
sns.set(style="white", context="notebook")

# LOSS
plt.figure(figsize=(8, 5))
plt.plot(grouped['epoch'], grouped['train_loss_mean'], label='Train Loss', color='blue')
plt.fill_between(grouped['epoch'],
                 grouped['train_loss_mean'] - grouped['train_loss_std'],
                 grouped['train_loss_mean'] + grouped['train_loss_std'],
                 color='blue', alpha=0.2)

plt.plot(grouped['epoch'], grouped['val_loss_mean'], label='Val Loss', color='orange')
plt.fill_between(grouped['epoch'],
                 grouped['val_loss_mean'] - grouped['val_loss_std'],
                 grouped['val_loss_mean'] + grouped['val_loss_std'],
                 color='orange', alpha=0.2)

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss (mean ± std)')
plt.legend()
sns.despine()
plt.tight_layout()
plt.show()

# ACCURACY
plt.figure(figsize=(8, 5))
plt.plot(grouped['epoch'], grouped['train_acc_mean'], label='Train Accuracy', color='green')
plt.fill_between(grouped['epoch'],
                 grouped['train_acc_mean'] - grouped['train_acc_std'],
                 grouped['train_acc_mean'] + grouped['train_acc_std'],
                 color='green', alpha=0.2)

plt.plot(grouped['epoch'], grouped['val_acc_mean'], label='Val Accuracy', color='red')
plt.fill_between(grouped['epoch'],
                 grouped['val_acc_mean'] - grouped['val_acc_std'],
                 grouped['val_acc_mean'] + grouped['val_acc_std'],
                 color='red', alpha=0.2)

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy (mean ± std)')
plt.legend()
sns.despine()
plt.tight_layout()
plt.show()

# Generative Adversarial Networks

## Training - GANs

In [ ]:
!python /content/mla_project/external/PyTorch-GAN/implementations/dcgan/dcgan.py \
    --generate_defect \
    --data_dir "/content/mla_project/images/original/Defects" \
    --batch_size 4 \
    --n_epochs 300 \
    --img_size 800 \
    --sample_interval 400 \
    --lr_g 0.00002 \
    --lr_d 0.0002

Namespace(n_epochs=300, batch_size=4, lr_g=2e-05, lr_d=0.0002, b1=0.5, b2=0.999, n_cpu=8, latent_dim=128, img_size=800, channels=1, sample_interval=400, data_dir='/content/mla_project/images/original/Defects', generate_defect=True)
Using GPU
Generating defect images...
Number of images: 47
/content/mla_project/external/PyTorch-GAN/implementations/dcgan/dcgan.py:155: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:78.)
  valid = Variable(Tensor(imgs.shape[0], 1).fill_(1.0), requires_grad=False)
[Epoch 0/300] [Batch 0/12] [D loss: 0.693308] [G loss: 0.693929]
[Epoch 0/300] [Batch 1/12] [D loss: 0.691083] [G loss: 0.693518]
[Epoch 0/300] [Batch 2/12] [D loss: 0.688530] [G loss: 0.696188]
[Epoch 0/300] [Batch 3/12] [D loss: 0.683807] [G loss: 0.700375]
[Epoch 0/300] [Batch 4/12] [D loss: 0.68124

### Generate new images using GANs

In [19]:
!python /content/mla_project/external/PyTorch-GAN/implementations/dcgan/generate.py \
      --model_path "/content/saved_models/generator_epoch_299.pth" \
      --img_size 800 \
      --num_images 5 \
      --output_dir "/content/generated_images"

Generated 2 images in /content/generated_images


In [17]:
!zip -r /content/generated_images.zip /content/generated_images

  adding: content/generated_images_2/ (stored 0%)
  adding: content/generated_images_2/image_3.png (deflated 5%)
  adding: content/generated_images_2/image_2.png (deflated 5%)
  adding: content/generated_images_2/image_4.png (deflated 5%)
  adding: content/generated_images_2/image_0.png (deflated 5%)
  adding: content/generated_images_2/image_1.png (deflated 5%)


### Upload model to HuggingFace

In [13]:
from huggingface_hub import login, create_repo, upload_file

# Effettua il login (ti verrà richiesto di inserire il token)
login()

# Carica un file nel repository
upload_file(
    path_or_fileobj="/content/saved_models/generator_epoch_299.pth",
    path_in_repo="Experiment_5_generator_epoch_299.pth",
    repo_id="MLinAppl/gan",
    repo_type="model"
)

generator_epoch_599.pth:   0%|          | 0.00/2.64G [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/MLinAppl/gan/commit/9d0b572d8b019d29197baec9ab9a19cfec6c1794', commit_message='Upload Experiment_4_generator_epoch_599.pth with huggingface_hub', commit_description='', oid='9d0b572d8b019d29197baec9ab9a19cfec6c1794', pr_url=None, repo_url=RepoUrl('https://huggingface.co/MLinAppl/gan', endpoint='https://huggingface.co', repo_type='model', repo_id='MLinAppl/gan'), pr_revision=None, pr_num=None)